# Technique 2

## Case 1: Earth to Asteroid Dionysus


In [1]:
import os
import sys
import platform

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)


In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

environment = dict(python=sys.version, executable=sys.executable,
                   platform=platform.platform(), machine=platform.machine(),
                   numpy=np.__version__, scipy=scipy.__version__, heyoka=hy.__version__,
                   threads=1, fast_cpus=sorted(fast_cpus), strict_cpu_pinning=False)
print(environment)

records_by_method = {}
results_by_method = {}

def save_trial(i, result, accepted, nfev):
    records.append(dict(trial=int(i), accepted=bool(accepted), nfev=int(nfev),
        status=int(result.status) if result is not None else None,
        residual_norm=float(np.linalg.norm(result.fun)) if result is not None else None,
        x=result.x.tolist() if result is not None else None))


{'python': '3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]', 'executable': '/Users/zhong/Documents/Codex/miniforge3/envs/gtoc13_heyoka7121_paper/bin/python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'machine': 'arm64', 'numpy': '2.5.3', 'scipy': '1.18.0', 'heyoka': '7.12.1', 'threads': 1, 'fast_cpus': [12, 13, 14, 15, 16, 17], 'strict_cpu_pinning': False}


In [3]:
import numpy as np

# =========================
# Setup (SI Units)
# =========================
AU2m = 149597870700 # JPL 
day2s = 86400.0     # JPL

tof = 3534.0*day2s # [s]

m0 = 4000.0 # [kg]
Tmax = 0.32 # [N]
Isp = 3000  # [s]

mu = 132712440018*1e9   # [m3∕s2]
g0 = 9.8065             # [m∕s2]

# Initial and terminal MEE (p, f, g, h, k, L); terminal L includes five revolutions.
mee0 = [149554268256.46957, -0.003766786954568226, 0.016286832739416963, -7.702048962437313e-06, 6.183156622292267e-07, 1.5955219194574601]

meef = [232433083985.70743, 0.1530290696088389, -0.5199481742007113, 0.01618310223871937, 0.11813952745106715, 33.763869726200596]

# =========================
# Output dimensional MEE in scientific notation
# =========================
print("mee0 (dimensional) =", np.array2string(
    np.asarray(mee0),
    formatter={"float_kind": lambda x: f"{x:.8e}"}
))
print("meef (dimensional) =", np.array2string(
    np.asarray(meef),
    formatter={"float_kind": lambda x: f"{x:.8e}"}
))

# =========================
# Nondimensionalize
# =========================
# Nondimension units
LUnit = AU2m
TUnit = np.sqrt(LUnit * LUnit * LUnit / mu)
MUnit = m0

VUnit = LUnit / TUnit # VUnit to m/s
AUnit = VUnit / TUnit # AUnit to m/s^2

# x0_nu, xf_nu, t0_nu, tf_nu, m0_nu, mu_nu, c1=Tmax_nu, c2=Isp_nu*g0_nu 
x0_nu = mee0
xf_nu = meef
t0_nu = 0.0
tf_nu = t0_nu+tof
m0_nu = m0
mu_nu = mu
Tmax_nu = Tmax
Isp_nu = Isp
g0_nu = g0

x0_nu[0] = x0_nu[0] / LUnit
xf_nu[0] = xf_nu[0] / LUnit

t0_nu = t0_nu / TUnit
tf_nu = tf_nu / TUnit
m0_nu = m0_nu / MUnit
mu_nu = mu_nu * TUnit * TUnit / LUnit / LUnit / LUnit
Tmax_nu = Tmax_nu / MUnit / AUnit
Isp_nu = Isp_nu / TUnit
g0_nu = g0_nu / AUnit

c1 = Tmax_nu
c2 = Isp_nu*g0_nu


mee0 (dimensional) = [1.49554268e+11 -3.76678695e-03 1.62868327e-02 -7.70204896e-06
 6.18315662e-07 1.59552192e+00]
meef (dimensional) = [2.32433084e+11 1.53029070e-01 -5.19948174e-01 1.61831022e-02
 1.18139527e-01 3.37638697e+01]


In [4]:
import twobody_MEE_hamilton_log

ta = twobody_MEE_hamilton_log.ta


In [5]:
q_penalty_coefficient = 10.0

# =========================
# Set parameters for heyoka integrator
# =========================

# =========================
# Define the shooting function
# =========================
def shooting(lam, eps = 1.0e-5, lambda0 = 1.0, out = False, Time_out = None):
    # lam[0:7] -> initial co-states [lp,lf,lg,lh,lk,lL,lm]
    # function uses global variables ta, t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    # mu_nu, c1, c2
    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam[:]

    result = ta.propagate_until(tf_nu)
    if result[0] == hy.taylor_outcome.step_limit:
        return np.full(7, np.nan, dtype=float)

    # Construct 7 equations: 6 terminal mee errors + 1 terminal co-state lm = 0
    retval = [
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13]
    ]

    if out:
        print("  Final Mass:", ta.state[6]*MUnit)

    if Time_out is not None:
        test = Time_out

    return retval
def shooting_augmented(lam_augmented, logeps0 = -0.,logepsf = -5., lambda0 = 1.0, out = False, Time_out = None):
    # Unknowns: seven initial co-states and q
    # function uses global variables ta, t0_nu, x0_nu, m0_nu, tf_nu, xf_nu, 
    # mu_nu, c1, c2

    # lam_augmented[0:7] -> initial co-states [lp,lf,lg,lh,lk,lL,lm]
    # lam_augmented[7]   -> normalized homotopy variable q
    # q = (logeps - logeps0) / (logepsf - logeps0)
    # so that:
    #   q = 0 -> eps = eps0
    #   q = 1 -> eps = epsf

    q = lam_augmented[7]    # Normalized homotopy variable q
    logeps = logeps0 + q * (logepsf - logeps0) # Recover the smoothing parameter from q
    eps = 10.0**logeps

    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam_augmented[:-1]
    result = ta.propagate_until(tf_nu)
    if result[0] == hy.taylor_outcome.step_limit:
        return np.full(8, np.nan, dtype=float)

    # Construct 8 equations:
    # 6 terminal MEE errors + lm(tf) = 0 + logeps = logepsf
    retval = [
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13],
        logeps - logepsf,     # log10(eps/epsf)
    ]
    
    if out:
        print("  Final Mass:", ta.state[6]*MUnit)

    if Time_out is not None:
        test = Time_out

    return retval

def shooting_augmented_gate(
    lam_augmented,
    logeps0 = 0.0,          # Initial log10(eps)
    logepsf = -5.0,         # Target log10(eps)
    rho0 = 1.0e-2,          # gate threshold
    delta_rho = 1.0e-8,     # regularization for rho
    gate_p = 2.0,           # gate exponent
    gate_kind = "gaussian", # gate option
    lambda0 = 1.0,
    out = False,
    Time_out = None
):
    
    # lam_augmented[0:7] -> initial co-states [lp,lf,lg,lh,lk,lL,lm]
    # lam_augmented[7]   -> normalized homotopy variable q
    # q = (logeps - logeps0) / (logepsf - logeps0)
    # so that:
    #   q = 0 -> eps = eps0
    #   q = 1 -> eps = epsf

    q = lam_augmented[7]    # Normalized homotopy variable q

    # Recover the smoothing parameter from q
    logeps = logeps0 + q * (logepsf - logeps0)
    eps = 10.0**logeps

    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = lam_augmented[:-1]
    result = ta.propagate_until(tf_nu)
    if result[0] == hy.taylor_outcome.step_limit:
        return np.full(8, np.nan, dtype=float)

    # Terminal residual vector Phi
    Phi = np.array([
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13]
    ], dtype=float)

    # Regularized RMS residual
    rho = np.sqrt(delta_rho**2 + np.mean(Phi * Phi))

    # Exponential and rational gate functions
    # gate switch
    if gate_kind == "gaussian":
        w = np.exp(- (rho / rho0) ** gate_p)
    elif gate_kind == "rational":
        w = 1.0 / (1.0 + (rho / rho0) ** gate_p)
    else:
        raise ValueError(f"Unknown gate_kind: {gate_kind}")

    # augmented residual:
    # when rho is large -> w ~ 0 -> q ~ 0 -> eps ~ eps0
    # when rho is small -> w ~ 1 -> q ~ 1 -> eps ~ epsf
    penalty_q = q_penalty_coefficient * (max(q - 1.0, 0.0)**2 - max(-q, 0.0)**2)
    retval = list(Phi) + [q - w + penalty_q]

    if out:
        # Output the smoothing parameter and residual norm
        print(f"  Final Mass: {ta.state[6]*MUnit}")
        print(f"  Final eps : {eps:.16e}  (logeps = {logeps:.8f}, q = {q:.8f}, rho = {rho:.3e})")

    if Time_out is not None:
        test = Time_out

    return retval

# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root
repeat_num_minpack  = 5000
# repeat_num_minpack  = 1

minpack_tol     = 1.0e-10
minpack_xtol    = 1.0e-10
minpack_maxfev  = 50000
minpack_factor  = 0.1
minpack_diag    = [1.0]*3 + [1.0]*3 + [1.0]*1
minpack_eps     = 0.
minpack_error   = 1.0e-7

np.set_printoptions(precision=16, suppress=True)


In [6]:

def prepare_timing():
    set_timing_priority()
    if sys.platform != 'darwin':
        return
    started = time.perf_counter()
    deadline = started + 0.25
    cpu = ctypes.c_size_t()
    while True:
        lib.pthread_cpu_number_np(ctypes.byref(cpu))
        if cpu.value in fast_cpus and time.perf_counter()-started >= 0.05:
            return
        if time.perf_counter() > deadline:
            raise RuntimeError('No Super-core scheduling during timing preparation.')


## Method 1: normal shooting method (eps = 1)


In [7]:
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

epsf = 1.0e0
# epsf = 1.0e-5
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = (np.random.random(7) - 0.5) * 2.0
    x0[-1] = np.random.random()          # λ_m ∈ (0,1)
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [8]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        # Call one more time with out = True to output terminal mass
        shooting(r['x'], epsf, lambda0, out=True, Time_out=None)
        # Save the best solution
        final_mass = ta.state[6]*MUnit
        if final_mass>global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)

records_by_method['Single_eps1'] = records


Convergence: 5000 original notebook trials evaluated in 16 single-thread workers.
Succeed to solve the problem!
Solved: 1825 in 5000 (0.3650), terminal mass: 2331.0362kg
Average number of function calls for Minpack: 83
Average number of function calls for convergent solution: 106
global_lambda: [-0.30097759179339134, -0.030778713868003012, 0.16798972162060208, -0.16078615184493333, -0.25784017022728317, -0.002175013788854442, 0.4425851974263541]


In [9]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_eps1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [10]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


11.1 s +- 120 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


4.17 s +- 69.5 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


7.24 s +- 128 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [13]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_eps1'] = dict(
    method='Single shooting, epsilon=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 1: normal shooting method (eps = 1e-5)


In [14]:
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

epsf = 1.0e-5
# epsf = 1.0e-5
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = (np.random.random(7) - 0.5) * 2.0
    x0[-1] = np.random.random()          # λ_m ∈ (0,1)
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [15]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        # Call one more time with out = True to output terminal mass
        shooting(r['x'], epsf, lambda0, out=True, Time_out=None)
        # Save the best solution
        final_mass = ta.state[6]*MUnit
        if final_mass>global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)

records_by_method['Single_eps1e5'] = records


Convergence: 5000 original notebook trials evaluated in 16 single-thread workers.
Succeed to solve the problem!
Solved: 444 in 5000 (0.0888), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 96
Average number of function calls for convergent solution: 171
global_lambda: [-0.34720984856791876, -0.05748250634662531, 0.1683148288139363, -0.221195805748356, -0.5423588777416889, 0.00037760452213291907, 0.3901673099509018]


In [16]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_eps1e5']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [17]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


34.2 s +- 210 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


6.5 s +- 30.4 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting,
        x0,
        args = (epsf, lambda0, False, None),   # order: lam, eps, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


27.9 s +- 198 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [20]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_eps1e5'] = dict(
    method='Single shooting, epsilon=1e-5', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1e5'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1e5']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 2: consecutive iterations on the smoothing parameter


In [21]:
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

# Continuation in eps: eps0 -> epsf
eps0 = 1.0e0        
epsf = 1.0e-5       

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# store eps / mf / lambda sequences for the global best solution
global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

# continuation/backtracking parameters (eps_next = eps_curr * d)
d0 = 0.1                 # initial step ratio (aggressive)
# d0 = 1.0e-5                 # initial step ratio (aggressive)
d_shrink = 0.8           # backtracking factor (used to INCREASE d so eps_next is closer to eps_curr)
d_recover = 0.8          # gentle recovery factor (after success, gradually DECREASE d back toward d0)
d_max = 0.999            # upper bound of d (avoid d==1 causing stagnation)
max_backtrack = 20       # max backtracking tries per step
max_total_steps = 300    # max accepted eps steps per continuation path (avoid infinite loops)

# wrap a single root() call to unify timing, nfev, residual, and success criterion
def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),  # (eps, lambda0, out, Time_out)
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    elapsed = time_end - time_start

    res = np.linalg.norm(r['fun'])
    ok = (r['success'] and res < minpack_error)
    return r, ok, res, elapsed

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = (np.random.random(7) - 0.5) * 2.0
    x0[-1] = np.random.random()          # λ_m ∈ (0,1)
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [22]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    r_next = None
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # per-outer-loop statistics (sum over all eps attempts, including failed backtracks)
    this_time = 0.0
    this_nfev = 0

    # continuation initialization
    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    # store eps / mf / lambda sequences for THIS successful path (accepted eps only)
    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 ONCE (outside inner loop)
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        # Record mf at eps0 (refresh ta.state explicitly)
        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())
    else:
        # Cannot solve even eps0 -> fail this random initial guess
        pass

    # ============================================================
    # 1) Continuation: advance eps until epsf (accepted eps only)
    # ============================================================
    if ok0:
        for step in range(max_total_steps):

            # Stop if target reached
            if eps_curr <= epsf:
                solved_this_outer = True

                # Update global best (also store full sequences)
                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            # Backtracking search for a feasible eps_next = eps_curr * d
            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                # Numerical safety: if eps_next does not change, force epsf
                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    # Accept eps_next
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    # Record mf/lambda at eps_curr (refresh ta.state explicitly)
                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    # gentle recovery of d (after success, gradually decrease d back toward d0)
                    d = max(d0, d * d_recover)

                    break
                else:
                    # Increase d to bring eps_next closer to eps_curr
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                # Too many backtracks and still cannot advance -> fail this random initial guess
                break

    # total_nfev/total_time include ALL eps attempts within this outer loop
    total_time += this_time
    total_nfev += this_nfev

    # only count success if eps reaches epsf; converge_nfev sums ALL nfev in that successful path
    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev

        print(
            f"counter: {count_solved}/{i+1} "
            f"steps(accepted): {len(eps_seq)} "
            f"nfev(sum): {this_nfev} "
            f"eps_end: {eps_seq[-1]:.1e} "
            f"mf_end: {mf_seq[-1]:.4f}kg",
            end='\n',
            flush=True
        )
    save_trial(i, r_next if ok0 and r_next is not None else r0, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")

    print("global_lambda (at epsf):", global_lambda)

    # Output the continuation history of the best solution
    print("global_eps_seq:", global_eps_seq)
    print("global_mf_seq:", global_mf_seq)
    print("global_lambda_seq length:", None if global_lambda_seq is None else len(global_lambda_seq))

records_by_method['Homotopy'] = records


Convergence: 5000 original notebook trials evaluated in 16 single-thread workers.
Succeed to solve the problem!
Solved: 1825 in 5000 (0.3650), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 129
Average number of function calls for convergent solution: 232
global_lambda (at epsf): [-0.3472098485654758, -0.057482506344575905, 0.16831482881708124, -0.22119580574273717, -0.5423588777389204, 0.00037760452204793307, 0.39016730995360394]
global_eps_seq: [1.0, 0.1, 0.010000000000000002, 0.0010000000000000002, 0.00010000000000000003, 1.0000000000000004e-05, 1e-05]
global_mf_seq: [2331.0362177007346, 2542.4990160619527, 2688.577595215548, 2715.053550392289, 2717.990385492868, 2718.2885919578703, 2718.2885919658506]
global_lambda_seq length: 7


In [23]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Homotopy']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [24]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

# Continuation in eps: eps0 -> epsf
eps0 = 1.0e0        
epsf = 1.0e-5       

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# store eps / mf / lambda sequences for the global best solution
global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

# continuation/backtracking parameters (eps_next = eps_curr * d)
d0 = 0.1                 # initial step ratio (aggressive)
# d0 = 1.0e-5                 # initial step ratio (aggressive)
d_shrink = 0.8           # backtracking factor (used to INCREASE d so eps_next is closer to eps_curr)
d_recover = 0.8          # gentle recovery factor (after success, gradually DECREASE d back toward d0)
d_max = 0.999            # upper bound of d (avoid d==1 causing stagnation)
max_backtrack = 20       # max backtracking tries per step
max_total_steps = 300    # max accepted eps steps per continuation path (avoid infinite loops)

# wrap a single root() call to unify timing, nfev, residual, and success criterion
def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),  # (eps, lambda0, out, Time_out)
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    elapsed = time_end - time_start

    res = np.linalg.norm(r['fun'])
    ok = (r['success'] and res < minpack_error)
    return r, ok, res, elapsed

# =========================
# Random initial guess loop
# =========================

for i in timing_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # per-outer-loop statistics (sum over all eps attempts, including failed backtracks)
    this_time = 0.0
    this_nfev = 0

    # continuation initialization
    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    # store eps / mf / lambda sequences for THIS successful path (accepted eps only)
    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 ONCE (outside inner loop)
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        # Record mf at eps0 (refresh ta.state explicitly)
        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())
    else:
        # Cannot solve even eps0 -> fail this random initial guess
        pass

    # ============================================================
    # 1) Continuation: advance eps until epsf (accepted eps only)
    # ============================================================
    if ok0:
        for step in range(max_total_steps):

            # Stop if target reached
            if eps_curr <= epsf:
                solved_this_outer = True

                # Update global best (also store full sequences)
                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            # Backtracking search for a feasible eps_next = eps_curr * d
            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                # Numerical safety: if eps_next does not change, force epsf
                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    # Accept eps_next
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    # Record mf/lambda at eps_curr (refresh ta.state explicitly)
                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    # gentle recovery of d (after success, gradually decrease d back toward d0)
                    d = max(d0, d * d_recover)

                    break
                else:
                    # Increase d to bring eps_next closer to eps_curr
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                # Too many backtracks and still cannot advance -> fail this random initial guess
                break

    # total_nfev/total_time include ALL eps attempts within this outer loop
    total_time += this_time
    total_nfev += this_nfev

    # only count success if eps reaches epsf; converge_nfev sums ALL nfev in that successful path
    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev


23.9 s +- 89.2 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [25]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

# Continuation in eps: eps0 -> epsf
eps0 = 1.0e0        
epsf = 1.0e-5       

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# store eps / mf / lambda sequences for the global best solution
global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

# continuation/backtracking parameters (eps_next = eps_curr * d)
d0 = 0.1                 # initial step ratio (aggressive)
# d0 = 1.0e-5                 # initial step ratio (aggressive)
d_shrink = 0.8           # backtracking factor (used to INCREASE d so eps_next is closer to eps_curr)
d_recover = 0.8          # gentle recovery factor (after success, gradually DECREASE d back toward d0)
d_max = 0.999            # upper bound of d (avoid d==1 causing stagnation)
max_backtrack = 20       # max backtracking tries per step
max_total_steps = 300    # max accepted eps steps per continuation path (avoid infinite loops)

# wrap a single root() call to unify timing, nfev, residual, and success criterion
def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),  # (eps, lambda0, out, Time_out)
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    elapsed = time_end - time_start

    res = np.linalg.norm(r['fun'])
    ok = (r['success'] and res < minpack_error)
    return r, ok, res, elapsed

# =========================
# Random initial guess loop
# =========================

for i in successful_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # per-outer-loop statistics (sum over all eps attempts, including failed backtracks)
    this_time = 0.0
    this_nfev = 0

    # continuation initialization
    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    # store eps / mf / lambda sequences for THIS successful path (accepted eps only)
    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 ONCE (outside inner loop)
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        # Record mf at eps0 (refresh ta.state explicitly)
        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())
    else:
        # Cannot solve even eps0 -> fail this random initial guess
        pass

    # ============================================================
    # 1) Continuation: advance eps until epsf (accepted eps only)
    # ============================================================
    if ok0:
        for step in range(max_total_steps):

            # Stop if target reached
            if eps_curr <= epsf:
                solved_this_outer = True

                # Update global best (also store full sequences)
                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            # Backtracking search for a feasible eps_next = eps_curr * d
            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                # Numerical safety: if eps_next does not change, force epsf
                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    # Accept eps_next
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    # Record mf/lambda at eps_curr (refresh ta.state explicitly)
                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    # gentle recovery of d (after success, gradually decrease d back toward d0)
                    d = max(d0, d * d_recover)

                    break
                else:
                    # Increase d to bring eps_next closer to eps_curr
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                # Too many backtracks and still cannot advance -> fail this random initial guess
                break

    # total_nfev/total_time include ALL eps attempts within this outer loop
    total_time += this_time
    total_nfev += this_nfev

    # only count success if eps reaches epsf; converge_nfev sums ALL nfev in that successful path
    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev


16.5 s +- 74.2 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [26]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

# Continuation in eps: eps0 -> epsf
eps0 = 1.0e0        
epsf = 1.0e-5       

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# store eps / mf / lambda sequences for the global best solution
global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

# continuation/backtracking parameters (eps_next = eps_curr * d)
d0 = 0.1                 # initial step ratio (aggressive)
# d0 = 1.0e-5                 # initial step ratio (aggressive)
d_shrink = 0.8           # backtracking factor (used to INCREASE d so eps_next is closer to eps_curr)
d_recover = 0.8          # gentle recovery factor (after success, gradually DECREASE d back toward d0)
d_max = 0.999            # upper bound of d (avoid d==1 causing stagnation)
max_backtrack = 20       # max backtracking tries per step
max_total_steps = 300    # max accepted eps steps per continuation path (avoid infinite loops)

# wrap a single root() call to unify timing, nfev, residual, and success criterion
def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),  # (eps, lambda0, out, Time_out)
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    elapsed = time_end - time_start

    res = np.linalg.norm(r['fun'])
    ok = (r['success'] and res < minpack_error)
    return r, ok, res, elapsed

# =========================
# Random initial guess loop
# =========================

for i in failed_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # per-outer-loop statistics (sum over all eps attempts, including failed backtracks)
    this_time = 0.0
    this_nfev = 0

    # continuation initialization
    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    # store eps / mf / lambda sequences for THIS successful path (accepted eps only)
    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 ONCE (outside inner loop)
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        # Record mf at eps0 (refresh ta.state explicitly)
        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())
    else:
        # Cannot solve even eps0 -> fail this random initial guess
        pass

    # ============================================================
    # 1) Continuation: advance eps until epsf (accepted eps only)
    # ============================================================
    if ok0:
        for step in range(max_total_steps):

            # Stop if target reached
            if eps_curr <= epsf:
                solved_this_outer = True

                # Update global best (also store full sequences)
                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            # Backtracking search for a feasible eps_next = eps_curr * d
            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                # Numerical safety: if eps_next does not change, force epsf
                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    # Accept eps_next
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    # Record mf/lambda at eps_curr (refresh ta.state explicitly)
                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    # gentle recovery of d (after success, gradually decrease d back toward d0)
                    d = max(d0, d * d_recover)

                    break
                else:
                    # Increase d to bring eps_next closer to eps_curr
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                # Too many backtracks and still cannot advance -> fail this random initial guess
                break

    # total_nfev/total_time include ALL eps attempts within this outer loop
    total_time += this_time
    total_nfev += this_nfev

    # only count success if eps reaches epsf; converge_nfev sums ALL nfev in that successful path
    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev


7.07 s +- 41.2 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [27]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Homotopy'] = dict(
    method='Stepwise', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Homotopy'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Homotopy']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 3: augmented shooting method


In [28]:
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = np.empty(8)

    x0[:7] = (np.random.random(7) - 0.5) * 2.0
    x0[6] = np.random.random()  # λ_m ∈ (0, 1)

    # x0[7] = logeps0
    x0[7] = 0.
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [29]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting_augmented,
        x0,
        # args = (logepsf, lambda0, False, None),   # order: lam, lamepsf, lambda0, out, Time_out
        args = (logeps0, logepsf, lambda0, False, None),   # order: lam, logeps0, logepsf, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        # Call one more time with out = True to output terminal mass
        # shooting_augmented(r['x'], logepsf, lambda0, out=True, Time_out=None)
        shooting_augmented(r['x'], logeps0, logepsf, lambda0, out=True, Time_out=None)
        # Save the best solution
        final_mass = ta.state[6]*MUnit
        if final_mass>global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)

records_by_method['Ungated'] = records


Convergence: 5000 original notebook trials evaluated in 16 single-thread workers.
Succeed to solve the problem!
Solved: 774 in 5000 (0.1548), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 115
Average number of function calls for convergent solution: 200
global_lambda: [-0.3472098485678685, -0.05748250634498168, 0.16831482881612775, -0.22119580573553072, -0.5423588777284873, 0.00037760452202091887, 0.3901673099463039, 1.0]


In [30]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Ungated']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [31]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting_augmented,
        x0,
        # args = (logepsf, lambda0, False, None),   # order: lam, lamepsf, lambda0, out, Time_out
        args = (logeps0, logepsf, lambda0, False, None),   # order: lam, logeps0, logepsf, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            'eps': minpack_eps,
        }
    )


22.1 s +- 54.7 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [32]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting_augmented,
        x0,
        # args = (logepsf, lambda0, False, None),   # order: lam, lamepsf, lambda0, out, Time_out
        args = (logeps0, logepsf, lambda0, False, None),   # order: lam, logeps0, logepsf, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            'eps': minpack_eps,
        }
    )


9.35 s +- 37.9 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [33]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # Random initial co-states
    x0 = initial_guess_pool[i].copy()

    # Use root's args parameter to pass fixed arguments
    r = root(
        shooting_augmented,
        x0,
        # args = (logepsf, lambda0, False, None),   # order: lam, lamepsf, lambda0, out, Time_out
        args = (logeps0, logepsf, lambda0, False, None),   # order: lam, logeps0, logepsf, lambda0, out, Time_out
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag + [1.],
            'eps': minpack_eps,
        }
    )


12.8 s +- 98.6 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [34]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Ungated'] = dict(
    method='Ungated', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Ungated'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Ungated']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 3: augmented shooting method with exponential gate (p = 1, A = 10)


In [35]:
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)

# parameters for the augmented residual
gate_option = 'gaussian'
gate_p = 1.0
rho0 = 0.2516222768095133

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = np.empty(8)

    x0[:7] = (np.random.random(7) - 0.5) * 2.0
    x0[6] = np.random.random()  # λ_m ∈ (0, 1)

    # start from q = 0 -> eps = eps0
    x0[7] = 0.0
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [36]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        # Evaluate the augmented shooting residual
        shooting_augmented_gate(
            r['x'],
            logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0,
            out = True,
            Time_out = None
        )

        # Save the best solution
        final_mass = ta.state[6]*MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['Gated_Exp_p1'] = records


Convergence: 5000 original notebook trials evaluated in 16 single-thread workers.
Succeed to solve the problem!
Solved: 1933 in 5000 (0.3866), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 134
Average number of function calls for convergent solution: 187
global_lambda: [-0.3472098485658984, -0.05748250634270226, 0.16831482881784873, -0.22119580573886102, -0.5423588777238709, 0.0003776045220228352, 0.3901673099511751, 0.9999999602479458]


In [37]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [38]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


22.2 s +- 39.7 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [39]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


14.9 s +- 66 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [40]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


7.28 s +- 17.4 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [41]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p1'] = dict(
    method='Option 1: Exp p=1, A=10', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 3: augmented shooting method with rational gate (p = 1, A = 10)


In [42]:
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)

# parameters for the augmented residual
gate_option = 'rational'
gate_p = 1.0
rho0 = 0.04303711287280454

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = np.empty(8)

    x0[:7] = (np.random.random(7) - 0.5) * 2.0
    x0[6] = np.random.random()  # λ_m ∈ (0, 1)

    # start from q = 0 -> eps = eps0
    x0[7] = 0.0
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [43]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        # Evaluate the augmented shooting residual
        shooting_augmented_gate(
            r['x'],
            logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0,
            out = True,
            Time_out = None
        )

        # Save the best solution
        final_mass = ta.state[6]*MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['Gated_Rational_p1'] = records


Convergence: 5000 original notebook trials evaluated in 16 single-thread workers.
Succeed to solve the problem!
Solved: 1891 in 5000 (0.3782), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 136
Average number of function calls for convergent solution: 194
global_lambda: [-0.34720984920689624, -0.05748250638877113, 0.16831482867626132, -0.2211958109640368, -0.5423588758109693, 0.0003776045409405216, 0.3901673085895316, 0.9999997048824505]


In [44]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [45]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


21.5 s +- 22.7 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [46]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


14 s +- 50.9 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [47]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


7.62 s +- 21.2 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [48]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p1'] = dict(
    method='Option 2: Rational p=1, A=10', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Method 3: augmented shooting method with rational gate (p = 2, A = 10)


In [49]:
# =========================
# Initialize solving time and best solution
# =========================
np.random.seed(322)

eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)

# parameters for the augmented residual
gate_option = 'rational'
gate_p = 2.0
rho0 = 0.09274286383257363

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8

# [general settings]:
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Random initial guess loop
# =========================

# =========================
# Common random initial guess pool
# =========================
np.random.seed(322)
initial_guess_pool = []
for i in range(repeat_num_minpack):
    x0 = np.empty(8)

    x0[:7] = (np.random.random(7) - 0.5) * 2.0
    x0[6] = np.random.random()  # λ_m ∈ (0, 1)

    # start from q = 0 -> eps = eps0
    x0[7] = 0.0
    initial_guess_pool.append(x0.copy())
initial_guess_pool = np.asarray(initial_guess_pool)


In [50]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    time_start = time.perf_counter()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    # Success criterion: solver converged AND residual below threshold
    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        # Evaluate the augmented shooting residual
        shooting_augmented_gate(
            r['x'],
            logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0,
            out = True,
            Time_out = None
        )

        # Save the best solution
        final_mass = ta.state[6]*MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)

# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print(f"Fail to solve the problem!")
else:
    print(f"Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    print("global_lambda:", global_lambda)
records_by_method['Gated_Rational_p2'] = records


Convergence: 5000 original notebook trials evaluated in 16 single-thread workers.
Succeed to solve the problem!
Solved: 1921 in 5000 (0.3842), terminal mass: 2718.2886kg
Average number of function calls for Minpack: 133
Average number of function calls for convergent solution: 185
global_lambda: [-0.3472098485669547, -0.057482506346618806, 0.16831482881962379, -0.2211958057400035, -0.5423588777219814, 0.00037760452201561826, 0.39016730995059645, 0.9999999999192503]


In [51]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p2']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [52]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


19.9 s +- 53.4 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [53]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


12.6 s +- 25.4 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [54]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # Random initial co-states + one normalized homotopy variable q
    x0 = initial_guess_pool[i].copy()

    r = root(
        shooting_augmented_gate,
        x0,
        # pass logeps0, logepsf, and gate parameters
        args = (logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method = 'hybr',
        tol = minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag+[1.],      
            # 'eps': minpack_eps,
            'eps': fd_eps,                  
        }
    )


7.33 s +- 47.2 ms per loop (mean +- std. dev. of 7 runs, 1 loop each)


In [55]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p2'] = dict(
    method='Option 3: Rational p=2, A=10', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


In [56]:

headings = ['Method', 'Success / 5000', 'Convergence [%]', 'Expected first [ms]',
            'Successful mean [ms]', 'Failed mean [ms]', 'All-trial mean [ms]']
table = ['| '+' | '.join(headings)+' |', '| '+' | '.join(['---']*len(headings))+' |']
for value in results_by_method.values():
    values = [value['method'], value['accepted'], f"{100.*value['accepted']/value['trials']:.2f}"]
    values += [f"{value[k]:.3f}" for k in ('first_ms','successful_ms','failed_ms','all_ms')]
    table.append('| '+' | '.join(map(str,values))+' |')
display(Markdown('\n'.join(table)))

| Method | Success / 5000 | Convergence [%] | Expected first [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] |
| --- | --- | --- | --- | --- | --- | --- |
| Single shooting, epsilon=1 | 1825 | 36.50 | 61.071 | 23.937 | 22.198 | 22.291 |
| Single shooting, epsilon=1e-5 | 444 | 8.88 | 770.116 | 138.301 | 61.655 | 68.386 |
| Stepwise | 1825 | 36.50 | 131.018 | 95.051 | 21.699 | 47.821 |
| Ungated | 774 | 15.48 | 285.513 | 112.661 | 30.742 | 44.197 |
| Option 1: Exp p=1, A=10 | 1933 | 38.66 | 114.873 | 73.916 | 24.417 | 44.410 |
| Option 2: Rational p=1, A=10 | 1891 | 37.82 | 113.786 | 70.486 | 25.242 | 43.034 |
| Option 3: Rational p=2, A=10 | 1921 | 38.42 | 103.599 | 62.699 | 24.506 | 39.803 |